In [1]:
import mediapipe as mp
import cv2
import pydirectinput

In [2]:
mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic
mp_pose = mp.solutions.pose


In [4]:
cap = cv2.VideoCapture(0)
cap.set(3, 560)
cap.set(4, 400)

with mp_holistic.Holistic(
    min_detection_confidence=0.5, 
    min_tracking_confidence=0.5
    ) as holistic:
    
    pose = 'idle'

    while cap.isOpened():
        success, img = cap.read()
        img = cv2.flip(img, 1)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        results = holistic.process(img)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        height, width, _ = img.shape
        y_mid = height//2

        try:
            wrist = results.pose_landmarks.landmark[
                mp_pose.PoseLandmark.LEFT_WRIST
            ]
            
            right_hand = (wrist.x * width, wrist.y * height)

            pose = 'move'
            if( right_hand[1] < y_mid ):
                pose = 'acc'
                pydirectinput.keyDown('right'),pydirectinput.keyUp('left')
            elif(right_hand[1]>y_mid):
                pose = 'brake'
                pydirectinput.keyDown('left')
                pydirectinput.keyUp('right')
            else:
                pose = "idle"
                pydirectinput.keyUp('left')
                pydirectinput.keyUp('right')
        except:
            pose = "No Detection"
        mp_drawing.draw_landmarks(img, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        cv2.putText(img, pose, (50, 50), cv2.FONT_HERSHEY_PLAIN, 2, (255, 255, 0), 2)
        cv2.line(img, (0, y_mid), (width, y_mid), (255, 0, 255), 2)
        cv2.imshow("Car Game", img)
        cv2.setWindowProperty("Car Game", cv2.WND_PROP_TOPMOST, 1)
            
        if cv2.waitKey(1) == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()
            
            

